# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All dataset structures are referenced by their `@id`, in accordance with FAIR principles.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL to the Croissant schema for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset name: {dataset.metadata.name}\nDataset description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# Display all available record sets and key fields within them, referencing by their @id

record_sets = dataset.metadata.record_sets()
print(f"Total record sets found: {len(record_sets)}\n")

for rset in record_sets:
    print(f"RecordSet @id: {rset['@id']}")
    print(f"  Name: {rset['name'] if 'name' in rset else 'N/A'}")
    print(f"  Description: {rset.get('description', 'N/A')}")
    fields = rset.get('field', [])
    # Ensure fields is a list
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        if isinstance(f, dict):
            fid = f.get('@id', None)
        else:
            fid = f
        print(f"    - {fid}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# We'll extract all record sets into DataFrames, keyed by their @id
all_record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    # Use generator to list all records, then to DataFrame
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet {record_set_id} with shape {df.shape}")
        else:
            print(f"RecordSet {record_set_id} has no records.")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# For demonstration, list the columns for the first non-empty DataFrame
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for RecordSet {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets could be loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field for analysis, filter records, normalize, and optionally group by another field. All fields are referenced by `@id` as shown above.

*If your dataset doesn't contain numeric fields or is not tabular, adapt the exploration accordingly.*

In [ ]:
# Pick the main record set (first loaded above)
record_set_id = main_record_set_id if dataframes else None

if record_set_id:
    df = dataframes[record_set_id].copy()
    
    # Try to auto-discover a numeric column (float or int)
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if not numeric_cols:
        print("No numeric columns detected for this record set.")
    else:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to guess a group/categorical field
        candidate_group_fields = df.select_dtypes(include='object').columns.tolist()
        group_field = None
        for c in candidate_group_fields:
            if df[c].nunique() < df.shape[0] and df[c].nunique() > 1:
                group_field = c
                break

        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} by group:")
            display(grouped_df.head())
        else:
            print("No suitable group/categorical field found for grouping.")
else:
    print("No tabular data was loaded to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields with `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped data exists
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

In this notebook, we loaded metadata and tabular records from the FAIR^2 dataset using the `mlcroissant` library, explored structure and fields by `@id`, performed basic EDA including filtering and normalization of a numeric field, optionally grouped by categorical variables, and visualized key metrics. This forms a reproducible template for FAIR Croissant-based data workflows.